# Forest / Non-Forest Notebook Prototype

This notebook adapts the AlphaEarth-style embedding workflow into a **binary forest/non-forest classifier** for the Gold Standard prototype repo.

It is designed to fit the current repository layout:

- Training and validation points come from a bundled demo dataset or from field pipeline outputs under `data/processed/`.
- Labels are read from one of `binary_label`, `lc_label`, `ipcc_class`, or `class`.
- The search area is taken from an `aoi` layer if present in `data/incoming/project.gpkg`; otherwise it falls back to a buffered envelope around the labeled points.
- Outputs are written to `output/forest_nonforest/`.

A small synthetic demo dataset is included at `data/demo/forest_nonforest_training.geojson` so the notebook can run end-to-end before real labels are ready.

## Prototype intent

This is a **starting point**, not yet a final Gold Standard production workflow. The goal is to prove that the repo can support a notebook-based EO classification workflow that later becomes a reusable module or CLI.

## Gold Standard notes

For a project-grade workflow, you would normally add:

- independent validation points and a formal confusion matrix
- conservative cloud/shadow handling
- date-specific maps for project start and about 10 years prior
- explicit forest definition logic and minimum mapping unit rules
- documented export products for eligibility assessment


In [ ]:
%%capture
# Run this cell to install notebook dependencies without cluttering cell output.
%pip install -q geopandas rioxarray xarray dask[distributed] scikit-learn aef-loader leafmap odc-geo pyproj matplotlib tabulate


In [ ]:
from pathlib import Path
import os
import subprocess

IN_COLAB = "google.colab" in str(get_ipython())
PREFERRED_COLAB_ROOT = Path("/content/gold-standard")
REPO_URL = "https://github.com/ulfboge/gold-standard.git"

if IN_COLAB:
    if not PREFERRED_COLAB_ROOT.exists():
        print(f"Cloning repo into {PREFERRED_COLAB_ROOT} ...")
        result = subprocess.run(
            ["git", "clone", REPO_URL, str(PREFERRED_COLAB_ROOT)],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            print(result.stdout)
            print(result.stderr)
            raise RuntimeError("Failed to clone the repository into /content/gold-standard")
    os.chdir(PREFERRED_COLAB_ROOT)
    print(f"Working directory set to {PREFERRED_COLAB_ROOT}")
else:
    print(f"Current working directory: {Path.cwd()}")

required_paths = [
    Path("docs/forest_nonforest_prototype.ipynb"),
    Path("data/demo/forest_nonforest_training.geojson"),
    Path("data/incoming"),
]

for path in required_paths:
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {path}")


In [ ]:
from pathlib import Path

import dask.array as da
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
import xarray as xr
from aef_loader import AEFIndex, DataSource, VirtualTiffReader
from aef_loader.utils import dequantize_aef, reproject_datatree
from dask.distributed import Client
from odc.geo.geobox import GeoBox
from pyproj import Transformer
from rasterio.features import shapes
from shapely.geometry import shape
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split

client = Client()
print(f"Dask dashboard: {client.dashboard_link}")


In [ ]:
YEAR = 2024
TARGET_CRS = "EPSG:3857"
RESOLUTION_M = 10
DEFAULT_BUFFER_M = 5000
FOREST_THRESHOLD = 0.5
ALLOWED_OBS_TYPES = {"ground_truth", "lc_training", "lc_validation"}

if IN_COLAB and PREFERRED_COLAB_ROOT.exists():
    REPO_ROOT = PREFERRED_COLAB_ROOT
else:
    REPO_ROOT = Path.cwd()

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DEMO_OBSERVATIONS_PATH = REPO_ROOT / "data" / "demo" / "forest_nonforest_training.geojson"
INCOMING_GPKG = REPO_ROOT / "data" / "incoming" / "project.gpkg"
OUTPUT_DIR = REPO_ROOT / "output" / "forest_nonforest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_WARNINGS = []
RUN_ERRORS = []
RUN_REPORT_PATH = OUTPUT_DIR / f"run_report_{YEAR}.md"


def add_warning(message: str):
    RUN_WARNINGS.append(message)
    print(f"WARNING: {message}")


def add_error(message: str):
    RUN_ERRORS.append(message)
    print(f"ERROR: {message}")


def write_run_report(status: str, extra_lines=None):
    lines = [
        f"# Forest / Non-Forest Run Report ({YEAR})",
        "",
        f"Status: `{status}`",
        "Dask used: `yes`",
        "How Dask is used: `dask.array` handles the large embedding raster in chunks and `Client()` starts a local scheduler. The scikit-learn model fit itself runs in-memory on the sampled training points.",
        f"Tile query years: `{(YEAR, YEAR)}`",
        f"Repository root: `{REPO_ROOT}`",
        f"Demo observations path: `{DEMO_OBSERVATIONS_PATH}`",
        f"Processed directory: `{PROCESSED_DIR}`",
        "",
        "## Warnings",
        "",
    ]
    if RUN_WARNINGS:
        lines.extend([f"- {message}" for message in RUN_WARNINGS])
    else:
        lines.append("- None")

    lines.extend(["", "## Errors", ""])
    if RUN_ERRORS:
        lines.extend([f"- {message}" for message in RUN_ERRORS])
    else:
        lines.append("- None")

    if extra_lines:
        lines.extend(["", "## Notes", ""])
        lines.extend(extra_lines)

    RUN_REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
    return RUN_REPORT_PATH


def latest_processed_geojson(processed_dir: Path) -> Path:
    matches = sorted(processed_dir.glob("run_*/project_enriched.geojson"))
    if not matches:
        raise FileNotFoundError("No processed GeoJSON found under data/processed/run_*/project_enriched.geojson")
    return matches[-1]


def resolve_observations_path() -> Path:
    if DEMO_OBSERVATIONS_PATH.exists():
        return DEMO_OBSERVATIONS_PATH
    add_warning(f"Bundled demo observations not found at {DEMO_OBSERVATIONS_PATH}; falling back to processed outputs.")
    return latest_processed_geojson(PROCESSED_DIR)


def normalize_binary_label(row: pd.Series):
    direct = row.get("binary_label")
    if pd.notna(direct):
        text = str(direct).strip().lower().replace("-", " ").replace("_", " ")
        if text in {"forest", "non forest"}:
            return text.replace(" ", "_")

    ipcc = row.get("ipcc_class")
    if pd.notna(ipcc):
        return "forest" if str(ipcc).strip().lower() == "forest land" else "non_forest"

    for field in ["lc_label", "class"]:
        value = row.get(field)
        if pd.isna(value):
            continue
        text = str(value).strip().lower().replace("-", " ").replace("_", " ")
        if "non forest" in text:
            return "non_forest"
        if "forest" in text:
            return "forest"
        if any(token in text for token in ["crop", "grass", "bare", "settlement", "wetland", "water"]):
            return "non_forest"

    return None


def load_labeled_points():
    observations_path = resolve_observations_path()
    points = gpd.read_file(observations_path)
    if "obs_type" in points.columns:
        points = points[points["obs_type"].isin(ALLOWED_OBS_TYPES)].copy()
    points["binary_label"] = points.apply(normalize_binary_label, axis=1)
    points = points.dropna(subset=["binary_label"]).copy()
    if points.empty:
        raise ValueError("No labeled points were found. Add forest and non-forest labels in the field data first.")
    return observations_path, points


def load_or_build_aoi(points_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if INCOMING_GPKG.exists():
        import fiona

        layers = set(fiona.listlayers(INCOMING_GPKG))
        if "aoi" in layers:
            aoi = gpd.read_file(INCOMING_GPKG, layer="aoi")
            return gpd.GeoDataFrame(geometry=[aoi.to_crs(points_gdf.crs).geometry.unary_union], crs=points_gdf.crs)

    add_warning("No `aoi` layer found in `data/incoming/project.gpkg`; using a buffered extent around labeled points.")
    buffered = points_gdf.to_crs(TARGET_CRS).buffer(DEFAULT_BUFFER_M).unary_union.envelope
    return gpd.GeoDataFrame(geometry=[buffered], crs=TARGET_CRS).to_crs(points_gdf.crs)


def build_geobox(aoi_gdf: gpd.GeoDataFrame, target_crs: str = TARGET_CRS, resolution: int = RESOLUTION_M):
    bbox = list(aoi_gdf.total_bounds)
    transformer = Transformer.from_crs(aoi_gdf.crs, target_crs, always_xy=True)
    x_min, y_min = transformer.transform(bbox[0], bbox[1])
    x_max, y_max = transformer.transform(bbox[2], bbox[3])
    geobox = GeoBox.from_bbox(
        bbox=(x_min, y_min, x_max, y_max),
        crs=target_crs,
        resolution=resolution,
    )
    return bbox, geobox


In [ ]:
observations_path, points = load_labeled_points()
aoi = load_or_build_aoi(points)
bbox_lonlat, geobox = build_geobox(aoi)

if observations_path == DEMO_OBSERVATIONS_PATH:
    add_warning("Using bundled demo observations instead of project field data.")

print(f"Observations source: {observations_path}")
print(f"Using bundled demo data: {observations_path == DEMO_OBSERVATIONS_PATH}")
print(f"Labeled points: {len(points)}")
print(points["binary_label"].value_counts())

if points["binary_label"].nunique() < 2:
    add_error("Need both forest and non-forest examples before training a binary classifier.")
    write_run_report("failed")
    raise ValueError("Need both forest and non-forest examples before training a binary classifier.")

points[[c for c in ["id", "obs_type", "binary_label", "lc_label", "ipcc_class", "class"] if c in points.columns]].head()


In [ ]:
try:
    index = AEFIndex(source=DataSource.SOURCE_COOP)
    await index.download()

    tiles = await index.query(
        bbox=bbox_lonlat,
        years=(YEAR, YEAR),
    )
    if not tiles:
        raise ValueError(f"No AlphaEarth tiles found for YEAR={YEAR} and bbox={bbox_lonlat}")

    async with VirtualTiffReader() as reader:
        tree = await reader.open_tiles_by_zone(tiles)

    combined = reproject_datatree(tree, target_geobox=geobox)
    embeddings = combined.embeddings.isel(time=0)
    embeddings_float = dequantize_aef(embeddings)
except Exception as exc:
    add_error(f"Embedding load failed: {type(exc).__name__}: {exc}")
    report_path = write_run_report(
        "failed",
        [
            "The failure happened while downloading, querying, or reprojecting AlphaEarth embeddings.",
            "If you are running in Colab, confirm the install cell completed and that the notebook is running from the expected folder.",
        ],
    )
    print(f"Saved run report to {report_path}")
    raise

embeddings_float


In [ ]:
points_proj = points.to_crs(TARGET_CRS)

training_embeddings = embeddings_float.sel(
    x=xr.DataArray(points_proj.geometry.x.to_list(), dims="sample"),
    y=xr.DataArray(points_proj.geometry.y.to_list(), dims="sample"),
    method="nearest",
).compute()

X = training_embeddings.transpose("sample", "band").values
y = (points["binary_label"] == "forest").astype(int).values

valid_mask = np.isfinite(X).all(axis=1)
X = X[valid_mask]
y = y[valid_mask]

print(f"Training samples after dropping invalid pixels: {len(y)}")

evaluation_rows = []

if len(y) >= 10 and np.bincount(y).min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y,
    )
    model = LogisticRegression(max_iter=2000, class_weight="balanced")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    evaluation_rows.append({
        "evaluation_split": "holdout",
        "n_samples": int(len(y_test)),
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "forest_precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "forest_recall": float(recall_score(y_test, y_pred, zero_division=0)),
    })
    print("Validation confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=["non_forest", "forest"]))
else:
    add_warning("Not enough balanced samples for a meaningful holdout validation; using a full-fit prototype model summary instead.")

model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X, y)
y_fit = model.predict(X)
evaluation_rows.append({
    "evaluation_split": "full_fit",
    "n_samples": int(len(y)),
    "accuracy": float(accuracy_score(y, y_fit)),
    "forest_precision": float(precision_score(y, y_fit, zero_division=0)),
    "forest_recall": float(recall_score(y, y_fit, zero_division=0)),
})
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df



In [ ]:
emb = embeddings_float.data
emb = da.moveaxis(emb, 0, -1)
ny, nx, nb = emb.shape
emb_2d = emb.reshape(-1, nb).rechunk({0: "auto", 1: -1})


def predict_forest_probability_block(block):
    valid = np.isfinite(block).all(axis=1)
    out = np.full(block.shape[0], np.nan, dtype=np.float32)
    if valid.any():
        out[valid] = model.predict_proba(block[valid])[:, 1].astype(np.float32)
    return out


forest_probability = emb_2d.map_blocks(
    predict_forest_probability_block,
    dtype=np.float32,
    drop_axis=1,
)

forest_probability_da = xr.DataArray(
    forest_probability.reshape(ny, nx),
    dims=["y", "x"],
    coords={
        "y": embeddings_float.coords["y"].values,
        "x": embeddings_float.coords["x"].values,
    },
    name="forest_probability",
)
forest_probability_da = forest_probability_da.rio.write_crs(embeddings_float.rio.crs)
forest_probability_da = forest_probability_da.rio.write_transform(embeddings_float.rio.transform())
forest_probability_da = forest_probability_da.rio.clip(aoi.to_crs(TARGET_CRS).geometry, aoi.to_crs(TARGET_CRS).crs, drop=True)

forest_mask = xr.where(forest_probability_da >= FOREST_THRESHOLD, 1, 0).astype("uint8")
forest_mask = forest_mask.rio.write_crs(forest_probability_da.rio.crs)
forest_mask = forest_mask.rio.write_transform(forest_probability_da.rio.transform())

probability_path = OUTPUT_DIR / f"forest_probability_{YEAR}.tif"
mask_path = OUTPUT_DIR / f"forest_nonforest_{YEAR}.tif"
forest_probability_da.rio.to_raster(probability_path)
forest_mask.rio.to_raster(mask_path)

mask_np = np.nan_to_num(forest_mask.values, nan=0).astype(np.uint8)
polygons = [
    shape(geom)
    for geom, value in shapes(mask_np, mask=mask_np == 1, transform=forest_mask.rio.transform())
    if value == 1
]
vector_path = OUTPUT_DIR / f"forest_polygons_{YEAR}.gpkg"
if polygons:
    forest_polygons = gpd.GeoDataFrame({"class": ["forest"] * len(polygons)}, geometry=polygons, crs=forest_mask.rio.crs)
    forest_polygons.to_file(vector_path, driver="GPKG")
    print(f"Saved forest polygons to {vector_path}")
else:
    add_warning("No forest polygons met the chosen threshold; vector export was skipped.")

print(f"Saved probability raster to {probability_path}")
print(f"Saved binary raster to {mask_path}")

forest_probability_da.plot(figsize=(8, 6), cmap="Greens")


In [ ]:
metrics_csv_path = OUTPUT_DIR / f"accuracy_summary_{YEAR}.csv"
metrics_md_path = OUTPUT_DIR / f"accuracy_summary_{YEAR}.md"
confusion_csv_path = OUTPUT_DIR / f"confusion_matrix_{YEAR}.csv"

evaluation_df.to_csv(metrics_csv_path, index=False)

fit_confusion = pd.DataFrame(
    confusion_matrix(y, y_fit),
    index=["actual_non_forest", "actual_forest"],
    columns=["pred_non_forest", "pred_forest"],
)
fit_confusion.to_csv(confusion_csv_path)

report_lines = [
    f"# Forest / Non-Forest Accuracy Summary ({YEAR})",
    "",
    f"Observations source: `{observations_path}`",
    f"Bundled demo data: `{observations_path == DEMO_OBSERVATIONS_PATH}`",
    f"Total labeled samples used: `{len(y)}`",
    f"Forest threshold: `{FOREST_THRESHOLD}`",
    "",
    "## Metrics",
    "",
    evaluation_df.to_markdown(index=False),
    "",
    "## Full-Fit Confusion Matrix",
    "",
    fit_confusion.to_markdown(),
    "",
    "## Output Files",
    "",
    f"- Probability raster: `{probability_path}`",
    f"- Binary raster: `{mask_path}`",
    f"- Metrics CSV: `{metrics_csv_path}`",
    f"- Confusion matrix CSV: `{confusion_csv_path}`",
]

run_report_notes = [
    f"Observations source: `{observations_path}`",
    f"Bundled demo data: `{observations_path == DEMO_OBSERVATIONS_PATH}`",
    f"Total labeled samples used: `{len(y)}`",
    f"Forest threshold: `{FOREST_THRESHOLD}`",
    "",
    "### Metrics",
    "",
    evaluation_df.to_markdown(index=False),
    "",
    "### Full-Fit Confusion Matrix",
    "",
    fit_confusion.to_markdown(),
    "",
    "### Output Files",
    "",
    f"- Probability raster: `{probability_path}`",
    f"- Binary raster: `{mask_path}`",
    f"- Metrics CSV: `{metrics_csv_path}`",
    f"- Confusion matrix CSV: `{confusion_csv_path}`",
]

if vector_path.exists():
    report_lines.append(f"- Forest polygons: `{vector_path}`")
    run_report_notes.append(f"- Forest polygons: `{vector_path}`")
else:
    run_report_notes.append("- Forest polygons were not exported.")

metrics_md_path.write_text("\n".join(report_lines), encoding="utf-8")
run_report_path = write_run_report("completed", run_report_notes)

print(f"Saved metrics CSV to {metrics_csv_path}")
print(f"Saved confusion matrix CSV to {confusion_csv_path}")
print(f"Saved Markdown summary to {metrics_md_path}")
print(f"Saved run report to {run_report_path}")

evaluation_df


## Next steps

To move this from prototype to project workflow, the next useful improvements are:

1. Add explicit `binary_label` values in field data so the class mapping is not heuristic.
2. Bring in an AOI polygon and date-specific analysis windows for Gold Standard eligibility checks.
3. Hold out independent `lc_validation` samples and archive the confusion matrix as a report artifact.
4. Convert the reusable parts of this notebook into a Python module under the repo, then keep the notebook for exploration and QA only.
